In [1]:

import mediapipe as mp
import numpy as np
import imutils
import cv2
import time
import os

os.chdir(r"D:\proj-python\Images_Vision")

In [20]:
hands=mp.solutions.hands.Hands(static_image_mode=False,max_num_hands=2,min_detection_confidence=0.5,min_tracking_confidence=0.5)
mp_draw=mp.solutions.drawing_utils

drawing_spec=mp_draw.DrawingSpec(thickness=2,circle_radius=4,color=(0,255,0))
connect_spec=mp_draw.DrawingSpec(thickness=2,circle_radius=2,color=(0,255,0))

like_img=cv2.resize(cv2.imread("like.png"),(200,200))
dislike_img=cv2.resize(cv2.imread("dislike.png"),(200,200))
fist_img=cv2.resize(cv2.imread("fist.png"),(200,200))
open_hand_img=cv2.resize(cv2.imread("open.png"),(200,200))
point_img=cv2.resize(cv2.imread("point.png"),(200,200))
peace_img=cv2.resize(cv2.imread("peace.png"),(200,200))
three_img=cv2.resize(cv2.imread("three.png"),(200,200))
pinch_img=cv2.resize(cv2.imread("pinch.png"),(200,200))
ok_img=cv2.resize(cv2.imread("ok.png"),(200,200))
call_img=cv2.resize(cv2.imread("call.png"),(200,200))
rock_img=cv2.resize(cv2.imread("rock.png"),(200,200))

images={
    "LIKE":like_img,
    "DISLIKE":dislike_img,
    "FIST":fist_img,
    "OPEN HAND":open_hand_img,
    "POINT":point_img,
    "PEACE":peace_img,
    "THREE":three_img,
    "PINCH":pinch_img,
    "OK":ok_img,
    "CALL":call_img,
    "ROCK":rock_img
}

cap=cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,1080)
cap.set(cv2.CAP_PROP_FPS,30)

prev_time=time.time()
prev_x=None
prev_y=None

while True:

    ret,frame=cap.read()

    if not ret:
        break

    h,w,c=frame.shape

    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    results=hands.process(rgb)

    gesture=None

    if results.multi_hand_landmarks:

        for hand_landmark,handedness in zip(results.multi_hand_landmarks,results.multi_handedness):

            label=handedness.classification[0].label
            lm=hand_landmark.landmark

            x_list=[int(p.x*w) for p in lm]
            y_list=[int(p.y*h) for p in lm]

            x_min=max(0,min(x_list)-20)
            x_max=min(w,max(x_list)+20)
            y_min=max(0,min(y_list)-20)
            y_max=min(h,max(y_list)+20)

            cv2.rectangle(frame,(x_min,y_min),(x_max,y_max),(0,255,0),2)
            cv2.putText(frame,label,(x_min,y_min-10),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,0),2)

            mp_draw.draw_landmarks(
                frame,
                hand_landmark,
                mp.solutions.hands.HAND_CONNECTIONS,
                drawing_spec,
                connect_spec
            )

            # FINGERS
            fingers=[]

            for tip in [8,12,16,20]:
                fingers.append(1 if lm[tip].y<lm[tip-2].y else 0)

            # CURRENT POSITION
            current_x=lm[1].x
            current_y=lm[1].y

            # MOVEMENT
            if prev_x is not None:

                dx=current_x-prev_x
                dy=current_y-prev_y

                if abs(dx)>0.02 and abs(dx)>abs(dy):
                   if dx>0:
                     gesture="SWIPE RIGHT"
                   else:
                     gesture="SWIPE LEFT"

                elif abs(dy)>0.02 and abs(dy)>abs(dx):
                   if dy>0:
                     gesture="MOVE DOWN"
                   else:
                     gesture="MOVE UP"

            prev_x=current_x
            prev_y=current_y

            # STATIC GESTURES
            if gesture is None:

                if abs(lm[4].x-lm[8].x)<0.04 and abs(lm[4].y-lm[8].y)<0.04 and fingers[1:]==[1,1,1]:
                    gesture="OK"

                elif abs(lm[4].x-lm[8].x)<0.05 and abs(lm[4].y-lm[8].y)<0.05:
                    gesture="PINCH"

                elif lm[4].y<lm[3].y and fingers==[0,0,0,0]:
                    gesture="LIKE"

                elif lm[4].y>lm[3].y and fingers==[0,0,0,0]:
                    gesture="DISLIKE"

                elif fingers==[0,0,0,0]:
                    gesture="FIST"

                elif fingers==[1,1,1,1]:
                    gesture="OPEN HAND"

                elif fingers==[1,0,0,0]:
                    gesture="POINT"

                elif fingers==[1,1,0,0]:
                    gesture="PEACE"

                elif fingers==[1,1,1,0]:
                    gesture="THREE"

                elif fingers==[0,0,0,1] and lm[4].y<lm[3].y:
                    gesture="CALL"

                elif fingers==[1,0,0,1]:
                    gesture="ROCK"

            # SHOW IMAGE
            if gesture in images:

                frame[20:220,20:220]=images[gesture]

                cv2.putText(
                    frame,
                    gesture,
                    (20,250),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,255,0),
                    2
                )

            else:
                cv2.putText(
                    frame,
                    gesture if gesture else "",
                    (20,250),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,255,0),
                    2
                )

    else:
        prev_x=None
        prev_y=None

    fps=1/(time.time()-prev_time)
    prev_time=time.time()

    cv2.putText(
        frame,
        f"FPS: {int(fps)}",
        (20,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,255,0),
        2
    )

    cv2.imshow("Hand Tracking",frame)

    if cv2.waitKey(1)&0xFF==ord("q"):
        break

cap.release()
hands.close()
cv2.destroyAllWindows()